In [ ]:
import os
import math
import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")

os.makedirs(TABLES_DIR, exist_ok=True)

FAKE_LABEL = 1
REAL_LABEL = 0

SEED = 42
rng = np.random.default_rng(SEED)

train_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), usecols=["full_text", "label"])
val_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "val.csv"), usecols=["full_text", "label"])
test_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), usecols=["full_text", "label"])

X_train = train_df["full_text"].astype(str).tolist()
y_train = train_df["label"].astype(int).to_numpy()

X_test = test_df["full_text"].astype(str).tolist()
y_test = test_df["label"].astype(int).to_numpy()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

tfidf_model = LinearSVC(class_weight="balanced")
tfidf_model.fit(X_train_tfidf, y_train)

tfidf_scores = tfidf_model.decision_function(X_test_tfidf)
tfidf_pred = tfidf_model.predict(X_test_tfidf)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BERT_DIR = os.path.join(MODELS_DIR, "bert_fulltext_trunc256")
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_DIR, use_fast=True)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_DIR)
bert_model.to(device)
bert_model.eval()

MAX_LEN = 256
BATCH_SIZE = 16

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = [str(x) for x in texts]
        self.labels = [int(x) for x in labels]
        self.tokenizer = tokenizer
        self.max_len = int(max_len)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

test_loader = DataLoader(
    NewsDataset(X_test, y_test.tolist(), bert_tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

all_logits = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out.logits.detach().cpu().numpy()

        all_logits.append(logits)
        all_labels.append(labels.detach().cpu().numpy())

bert_logits = np.concatenate(all_logits, axis=0)
bert_true = np.concatenate(all_labels, axis=0)
bert_probs = torch.softmax(torch.tensor(bert_logits), dim=1).numpy()
bert_scores = bert_probs[:, FAKE_LABEL]
bert_pred = np.argmax(bert_logits, axis=1)

def compute_metrics(y_true, y_pred, y_score, model_name):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    p_fake, r_fake, f1_fake, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=FAKE_LABEL, zero_division=0
    )
    roc_auc = roc_auc_score((y_true == FAKE_LABEL).astype(int), y_score)
    pr_auc = average_precision_score((y_true == FAKE_LABEL).astype(int), y_score)
    return {
        "model": model_name,
        "split": "test",
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "mcc": float(mcc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "precision_fake(1)": float(p_fake),
        "recall_fake(1)": float(r_fake),
        "f1_fake(1)": float(f1_fake),
        "roc_auc_fake(1)": float(roc_auc),
        "pr_auc_fake(1)": float(pr_auc),
    }

tfidf_metrics = compute_metrics(y_test, tfidf_pred, tfidf_scores, "TFIDF_LinearSVC")
bert_metrics = compute_metrics(bert_true, bert_pred, bert_scores, "BERT_trunc256")

metrics_df = pd.DataFrame([tfidf_metrics, bert_metrics])
metrics_path = os.path.join(TABLES_DIR, "enhanced_metrics_test.csv")
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)
print(metrics_df)

def bootstrap_ci(y_true, y_pred, y_score, n_boot=1000, seed=42):
    r = np.random.default_rng(seed)
    n = len(y_true)
    stats = {
        "accuracy": [],
        "macro_f1": [],
        "f1_fake(1)": [],
        "roc_auc_fake(1)": [],
        "pr_auc_fake(1)": [],
        "mcc": [],
        "balanced_accuracy": [],
    }
    y_bin = (y_true == FAKE_LABEL).astype(int)

    for _ in range(n_boot):
        idx = r.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        ys = y_score[idx]
        yb = y_bin[idx]

        stats["accuracy"].append(accuracy_score(yt, yp))
        stats["balanced_accuracy"].append(balanced_accuracy_score(yt, yp))
        stats["mcc"].append(matthews_corrcoef(yt, yp))
        stats["macro_f1"].append(f1_score(yt, yp, average="macro"))
        stats["f1_fake(1)"].append(f1_score((yt == FAKE_LABEL).astype(int), (yp == FAKE_LABEL).astype(int)))
        stats["roc_auc_fake(1)"].append(roc_auc_score(yb, ys))
        stats["pr_auc_fake(1)"].append(average_precision_score(yb, ys))

    out = {}
    for k, arr in stats.items():
        a = np.asarray(arr, dtype=float)
        out[k] = {
            "mean": float(np.mean(a)),
            "ci95_low": float(np.quantile(a, 0.025)),
            "ci95_high": float(np.quantile(a, 0.975)),
        }
    return out

N_BOOT = 1000

tfidf_ci = bootstrap_ci(y_test, tfidf_pred, tfidf_scores, n_boot=N_BOOT, seed=SEED)
bert_ci = bootstrap_ci(bert_true, bert_pred, bert_scores, n_boot=N_BOOT, seed=SEED)

rows = []
for model_name, ci in [("TFIDF_LinearSVC", tfidf_ci), ("BERT_trunc256", bert_ci)]:
    for metric_name, d in ci.items():
        rows.append({
            "model": model_name,
            "metric": metric_name,
            "mean": d["mean"],
            "ci95_low": d["ci95_low"],
            "ci95_high": d["ci95_high"],
            "n_boot": N_BOOT,
        })

ci_df = pd.DataFrame(rows)
ci_path = os.path.join(TABLES_DIR, "bootstrap_ci_test.csv")
ci_df.to_csv(ci_path, index=False)
print("Saved:", ci_path)
print(ci_df.head(12))